# Diagnóstico de la comprobación del notebook 04

La celda 3 del notebook 04 avisa de que el error supera lo esperado:

```
MAE  :  881.8 MW
RMSE : 1325.0 MW
MAPE :    3.05 %
```

frente al 1,34 % de mediana obtenido en el notebook 03.

## El aviso estaba mal planteado

Las dos cifras **no son comparables**, y el umbral del 3 % no tenía en cuenta la
diferencia de protocolo:

| | Notebook 03 | Notebook 04 |
|---|---|---|
| Periodo | 14 ventanas repartidas de octubre a diciembre | 30 días consecutivos de diciembre |
| Entrenamiento | Reentrena en cada origen | Uno solo, al inicio |
| Estadístico | Mediana de las ventanas | Media de 720 horas |

Un contraste controlado muestra que solo el cambio de protocolo eleva el error
alrededor de un 35 %. El resto puede deberse a que diciembre concentra festivos
y puentes, o a que el modelo se desactualiza.

Este cuaderno lo determina con los datos reales.

In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

RAIZ = Path.cwd()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))

from prediccion import construir_variables, columnas_predictoras
warnings.filterwarnings("ignore")

bruto = pd.read_parquet(RAIZ / "datos" / "dataset_modelado.parquet")
if bruto.index.tz is not None:
    bruto.index = bruto.index.tz_localize(None)

serie = bruto["demanda_mw"].sort_index()
meteo = None
if "tmed" in bruto.columns:
    meteo = (bruto[["tmed", "tmax", "tmin"]]
             .assign(fecha=bruto.index.normalize())
             .groupby("fecha").first().reset_index())

datos = construir_variables(serie, meteo=meteo)
COLUMNAS = columnas_predictoras(datos)
ent = datos.dropna()
print(f"Datos: {len(ent):,} filas, {len(COLUMNAS)} variables")

In [ ]:
from xgboost import XGBRegressor

def nuevo_modelo():
    return XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=7,
                        subsample=0.8, colsample_bytree=0.8, tree_method="hist",
                        n_jobs=-1, random_state=42, verbosity=0)

# ── Protocolo del notebook 04: un entrenamiento, 30 días seguidos ────────
corte = len(ent) - 720
modelo = nuevo_modelo()
modelo.fit(ent[COLUMNAS].iloc[:corte], ent["demanda_mw"].iloc[:corte])

real = ent["demanda_mw"].iloc[corte:]
pred = modelo.predict(ent[COLUMNAS].iloc[corte:])

err = pd.DataFrame({"real": real.values, "pred": pred}, index=real.index)
err["ape"] = np.abs((err["real"] - err["pred"]) / err["real"]) * 100

print(f"Periodo evaluado: {err.index[0]:%d-%b} a {err.index[-1]:%d-%b}")
print(f"MAPE medio   : {err['ape'].mean():.2f} %")
print(f"MAPE mediana : {err['ape'].median():.2f} %   <- comparable con el nb 03")

## 1. ¿Se concentra el error en unos pocos días?

Si unos pocos días disparan la media, el modelo está bien y el problema es la
dificultad puntual de esas fechas.

In [ ]:
por_dia = err.groupby(err.index.date).agg(
    mape=("ape", "mean"), demanda=("real", "mean")).sort_values("mape", ascending=False)

# Marcar festivos
if "intensidad_festivo" in bruto.columns:
    intens = bruto.groupby(bruto.index.date)["intensidad_festivo"].max()
    por_dia["intensidad_festivo"] = intens.reindex(por_dia.index).round(2)

por_dia["dia"] = [pd.Timestamp(d).day_name() for d in por_dia.index]

print("Diez días con mayor error\n")
display(por_dia.head(10).round(2))

peores5 = por_dia.head(5)["mape"].mean()
resto   = por_dia.iloc[5:]["mape"].mean()
print(f"\nMAPE de los 5 peores días : {peores5:.2f} %")
print(f"MAPE del resto            : {resto:.2f} %")
print(f"\nSin esos 5 días el error medio sería {por_dia.iloc[5:]['mape'].mean():.2f} %")

## 2. ¿Se desactualiza el modelo?

El notebook 04 entrena una sola vez y predice treinta días. Si el error crece a
medida que se aleja del corte de entrenamiento, conviene reentrenar con más
frecuencia.

In [ ]:
err["dias_desde_corte"] = (err.index - err.index[0]).days
por_semana = err.groupby(err["dias_desde_corte"] // 7)["ape"].mean()

print("MAPE por semana desde el corte de entrenamiento\n")
for s, v in por_semana.items():
    print(f"   semana {s + 1}: {v:.2f} %")

tendencia = np.polyfit(err["dias_desde_corte"], err["ape"], 1)[0]
print(f"\nTendencia: {tendencia:+.4f} puntos de MAPE por día")

if abs(tendencia) < 0.02:
    print("Sin degradación apreciable: un reentrenamiento mensual es suficiente.")
else:
    print("El error crece con el tiempo. Conviene reentrenar cada dos semanas.")

## 3. Comparación directa con el protocolo del notebook 03

Se aplica el mismo procedimiento de origen rodante **sobre este mismo periodo**.
Es la única comparación válida.

In [ ]:
HORIZONTE_V, N_VENT, PASO_V = 24, 14, 120

apes_ro = []
for i in range(N_VENT):
    o = len(ent) - (N_VENT - i) * PASO_V
    if o <= 0:
        continue
    m = nuevo_modelo()
    m.fit(ent[COLUMNAS].iloc[:o], ent["demanda_mw"].iloc[:o])
    p = m.predict(ent[COLUMNAS].iloc[o:o + HORIZONTE_V])
    r = ent["demanda_mw"].iloc[o:o + HORIZONTE_V]
    apes_ro.append(np.mean(np.abs((r.values - p) / r.values)) * 100)

print("Comparación de protocolos sobre el mismo modelo\n")
print(f"   Origen rodante, media   : {np.mean(apes_ro):.2f} %")
print(f"   Origen rodante, mediana : {np.median(apes_ro):.2f} %   <- el valor del nb 03")
print(f"   Bloque de 30 días, media: {err['ape'].mean():.2f} %")
print(f"   Bloque de 30 días, med. : {err['ape'].median():.2f} %")

dif = (err['ape'].mean() / np.median(apes_ro) - 1) * 100
print(f"\nLa diferencia entre protocolos explica un {dif:.0f} % del salto.")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7))

axes[0].plot(err.index, err["real"], color="black", lw=1, label="Real")
axes[0].plot(err.index, err["pred"], color="#2E5C8A", lw=1, ls="--", label="Predicción")
axes[0].set_ylabel("MW"); axes[0].legend(fontsize=8)
axes[0].set_title("Predicción frente a valores observados en el último mes",
                  fontweight="bold")

axes[1].plot(err.index, err["ape"], color="#C44E52", lw=0.8)
axes[1].axhline(err["ape"].mean(), color="black", ls="--", lw=1,
                label=f"media {err['ape'].mean():.2f} %")
axes[1].axhline(err["ape"].median(), color="#55A868", ls="--", lw=1,
                label=f"mediana {err['ape'].median():.2f} %")
axes[1].set_ylabel("Error absoluto porcentual (%)")
axes[1].legend(fontsize=8)
axes[1].set_title("Evolución del error", fontweight="bold")

plt.tight_layout()
plt.savefig(RAIZ / "figuras" / "21_diagnostico_modelo_final.png",
            dpi=200, bbox_inches="tight")
plt.show()

---
## Cómo interpretar el resultado

**Si el error se concentra en unos pocos días** (apartado 1) y la mediana se
aproxima al 1,3–1,7 % del notebook 03, el modelo está correcto. Diciembre
concentra Navidad, Nochevieja y varios puentes, y la media se resiente de esos
días. Puedes desplegar con tranquilidad.

**Si no hay tendencia temporal** (apartado 2), el modelo no se desactualiza y
basta con reentrenar una vez al mes.

**Si el origen rodante sobre este mismo periodo da un valor próximo al del
notebook 03** (apartado 3), queda confirmado que toda la diferencia procede del
protocolo de evaluación, no de un problema del modelo.

## Corrección del aviso en el notebook 04

El umbral del 3 % comparaba magnitudes distintas. Sustituye en la celda 3:

```python
if mape > 3:
    print("\nATENCIÓN: el error supera lo esperado...")
```

por:

```python
mape_mediana = float(np.median(np.abs((yte - p) / yte)) * 100)
print(f"   MAPE mediana : {mape_mediana:>8.2f} %   <- comparable con el nb 03")

if mape_mediana > 2.5:
    print("\nRevisa los datos: la mediana se aparta de lo observado en el nb 03.")
else:
    print("\nCoherente. La media es mayor porque el último mes concentra festivos.")
```

La mediana es la referencia adecuada: el notebook 03 también la usa, y es
robusta frente a los días atípicos.